# Chapter 4 — GIC: Generic Interrupt Controller

## Concept
GICv3 architecture: Distributor (GICD) for global SPI routing, Redistributors
(GICR) per CPU for PPI/SGI/LPI, CPU Interfaces (ICC_* system registers).
Interrupt types: SGI (0–15), PPI (16–31), SPI (32–1019), LPI (8192+).
ITS (Interrupt Translation Service) maps device/event IDs to LPIs for MSI-X.
QEMU `virt` emulates GICv3 with full ITS when `-machine virt` is used.

## Lab Objectives
1. Confirm GICv3 driver is present in `/proc/interrupts`.
2. Generate network traffic to increment the virtio-net SPI counter.
3. Parse ACPI MADT for ITS entry (`acpidump` + `acpixtract`).
4. Verify no spurious interrupts accumulate at idle.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# virtio-net required for IRQ counter test
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=["-netdev", "user,id=net0", "-device", "virtio-net-pci,netdev=net0"],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Read /proc/interrupts — confirm GICv3 ────────────────────────────
interrupts = sc.run_command("cat /proc/interrupts", timeout=15)
print(interrupts[:3000])


In [ ]:
# ── Step 2: Identify GICv3 header line ───────────────────────────────────────
gic_header = sc.run_command(
    "head -1 /proc/interrupts", timeout=10
)
print("Header:", gic_header)


In [ ]:
# ── Step 3: Generate loopback traffic to trigger virtio-net interrupts ────────
# Ping loopback — any network traffic will fire the virtio-net SPI.
sc.run_command("ping -c 5 127.0.0.1 > /dev/null 2>&1 &", timeout=10)
time.sleep(6)

interrupts_after = sc.run_command("cat /proc/interrupts", timeout=15)
print(interrupts_after[:3000])


In [ ]:
# ── Step 4: Grep virtio interrupt lines ──────────────────────────────────────
virtio_irqs = sc.run_command(
    "grep -i 'virtio' /proc/interrupts || echo 'no virtio IRQ lines'",
    timeout=10
)
print("virtio IRQ lines:\n", virtio_irqs)


In [ ]:
# ── Step 5: Check ACPI MADT for ITS entry (requires acpidump) ────────────────
acpi_check = sc.run_command(
    "which acpidump 2>/dev/null || echo 'acpidump not found'", timeout=5
)
print(acpi_check)

if "acpidump" in acpi_check and "not found" not in acpi_check:
    madt_raw = sc.run_command(
        "sudo acpidump -n MADT 2>/dev/null | head -60 || "
        "acpidump -n MADT 2>/dev/null | head -60",
        timeout=20
    )
    print("MADT dump (first 60 lines):\n", madt_raw)
else:
    # Fallback: check ACPI tables via /sys/firmware/acpi/tables
    madt_raw = sc.run_command(
        "ls /sys/firmware/acpi/tables/ 2>/dev/null || echo 'no ACPI tables'",
        timeout=10
    )
    print("ACPI tables via sysfs:", madt_raw)


In [ ]:
# ── Step 6: Count spurious interrupts in idle state ──────────────────────────
import re as _re
spuri_lines = [l for l in interrupts_after.splitlines()
               if _re.search(r'[Ss]purious|ERR', l)]
print(f"Spurious/ERR interrupt lines: {len(spuri_lines)}")
if spuri_lines:
    for l in spuri_lines:
        print(" ", l)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert_contains(interrupts, r"GICv3|gic-v3|arm-gic",
                "GICv3 driver entry in /proc/interrupts",
                action="Check -machine virt; GICv3 requires virt board")

assert_contains(interrupts, r"\d+:\s+\d+",
                "/proc/interrupts has numeric interrupt lines",
                action="Guest kernel not reading interrupt counters")

assert_contains(virtio_irqs, r"virtio|eth|net",
                "virtio-net or similar interrupt line present",
                action="Add -device virtio-net-pci to QEMU launch args")

# Check counter changed after traffic (at least one virtio line has count > 0)
import re as _re
counts = _re.findall(r'virtio.*?(\d+)', interrupts_after)
nonzero = [c for c in counts if int(c) > 0]
assert_true(len(nonzero) > 0,
            "virtio interrupt counter is non-zero after traffic",
            detail=f"non-zero counts: {nonzero[:5]}",
            action="Generate more traffic; virtio-net must be attached")

# ITS / MADT
assert_true(
    "ITS" in madt_raw or "GIC" in madt_raw or "APIC" in madt_raw
    or "firmware/acpi" in madt_raw,
    "ACPI MADT or ACPI tables accessible",
    detail=madt_raw[:100],
    action="Install acpica-tools: brew install acpica",
)

# No spurious interrupts
assert_equal(len(spuri_lines), 0,
             "Zero spurious interrupt lines at idle",
             action="Investigate interrupt handler — potential GIC misconfiguration")


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| GICv3 in /proc/interrupts | Present | GICv3 driver registered |
| Numeric interrupt lines | Present | Counters readable |
| virtio-net IRQ line | Present | PCI device attached |
| virtio counter > 0 post traffic | Yes | SPI routed correctly |
| ACPI MADT / tables accessible | Yes | UEFI ACPI path |
| Spurious interrupts | 0 | Clean interrupt routing |

> **Fidelity**: GICv3 + ITS emulation in QEMU is functionally accurate.
> Hardware LPI table initialisation differs slightly — QEMU auto-initialises
> the LPI pending table; real GICv3 requires explicit BASER programming.

## Next Steps
→ **Chapter 5**: PSCI — boot a 4-vCPU VM and verify all secondaries come online.
